In [1]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

from ydata_profiling import ProfileReport
import plotly.express as px
from pathlib import Path
datadir = Path("../../data")
reportdir = Path("../../reports")

import pandas as pd
report = False

/home/gustavo/miniconda3/envs/panvel/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def gen_profile(df, title, filepath):
    profile = ProfileReport(df, title=title)
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    profile.to_file(filepath)
    return profile

dfs = {}
for file in datadir.rglob("*.parquet"):
    dfs[file.stem] = pd.read_parquet(file)
    if report:
        prof_auto = gen_profile(dfs[file.stem], file.stem, reportdir.joinpath(f"{file.stem}.html"))

In [3]:
for key in dfs.keys():
    print(key)
    print(dfs[key].columns)

filiais
Index(['codigo_filial', 'faixa_vida', 'localidade', 'uf',
       'tipo_estabelecimento', 'delivery', 'metragem_area_venda',
       'panvel_clinic', 'estacionamento', 'atendimento_24_horas'],
      dtype='object')
metas
Index(['codigo_filial', 'data_meta_venda', 'meta_n_med', 'meta_med',
       'valor_meta_venda'],
      dtype='object')
devolucoes
Index(['codigo_filial', 'data_devolucao', 'categoria_gerencial', 'quantidade',
       'valor_devolucao'],
      dtype='object')
vendas
Index(['codigo_documento_saida', 'codigo_filial', 'data_emissao',
       'categoria_gerencial', 'quantidade', 'faturamento'],
      dtype='object')


In [4]:
dfd = dfs["devolucoes"].copy()
dfv = dfs["vendas"].copy()
dff = dfs["filiais"].copy()
dfm = dfs["metas"].copy()

In [5]:
ticket_med = dfv.groupby(by="codigo_filial").agg({
    "quantidade": "sum",
    "faturamento": "sum"
})

ticket_med["ticket_medio"] = ticket_med["faturamento"].astype(int) / ticket_med["quantidade"].astype(int)
ticket_med = ticket_med.reset_index()
ticket_med = ticket_med.rename(columns={"faturamento":"faturamento_total", "quantidade":"quantidade_total"})

Adicionando os valores de ticket medio e faturamento total e quantidade total de vendas ao `filiais`

In [6]:
dff = pd.merge(left=ticket_med, right=dff, how="inner", on="codigo_filial")
tmp = gen_profile(dff, "Filiais com Faturamento", reportdir.joinpath("filiais_ticket.html"))

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 1071.34it/s]


In [11]:
dfv['data_emissao'] = pd.to_datetime(dfv['data_emissao']).dt.date
dfv_filial_day = dfv.groupby(by=["codigo_filial", "data_emissao"]).agg({
    "faturamento": ["sum", "mean"],
    "quantidade": ["sum", "mean"]
})

dfv_day = dfv.groupby(by=["data_emissao"]).agg({
    "faturamento": ["sum", "mean"],
    "quantidade": ["sum", "mean"]
})
dfv_day.columns = ['_'.join(col).strip('_') for col in dfv_day.columns.values]

dfv_cat_day = dfv.groupby(by=["data_emissao", "categoria_gerencial"]).agg({
    "faturamento": ["sum", "mean"],
    "quantidade": ["sum", "mean"]
})
dfv_cat_day.columns = ['_'.join(col).strip('_') for col in dfv_cat_day.columns.values]

In [8]:
fig = px.area(dfv_day, x=dfv_day.index, y="faturamento_sum", line_shape="spline")
fig.update_xaxes(rangeslider_visible=True)
fig.show()

In [20]:

dfv_day_reset = dfv_day.reset_index()

df_plot = dfv_day_reset.melt(
    id_vars=['data_emissao'], 
    value_vars=['faturamento_sum', 'faturamento_mean'],
    var_name='metrica', 
    value_name='valor'
)

fig = px.area(
    df_plot, 
    x="data_emissao", 
    y="valor", 
    facet_col="metrica",
    color="metrica",
    line_shape="spline",
)
fig.update_yaxes(matches=None)
fig.update_xaxes(rangeslider_visible=True)

fig.show()

In [13]:
dfv_day_reset = dfv_cat_day.reset_index()

fig_cat = px.area(
    dfv_day_reset,
    x='data_emissao',
    y='faturamento_sum',
    color='categoria_gerencial',
    line_shape='spline',
    labels={
        'faturamento_sum': 'Faturamento Total', 
        'data_emissao': 'Data da Venda', 
        'categoria_gerencial': 'Categoria'
    },
    facet_col="categoria_gerencial"
)
fig_cat.update_xaxes(rangeslider_visible=True)
fig_cat.show()

In [23]:
fig_box = px.box(
    dff,
    x='tipo_estabelecimento',
    y='faturamento_total',
    color='panvel_clinic', 
)
fig_box.show()